In [1]:
!pip install facenet-pytorch


#Mount Google Drive & Set Paths



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/A1/dataset.zip'
extract_to = '/content/drive/MyDrive/A1/dataset/'

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    all_files = z.namelist()
    total = len(all_files)
    print(f'Total files in zip: {total}')

    for i, file in enumerate(all_files):
        z.extract(file, extract_to)
        if i % 5000 == 0:
            print(f'Progress: {i}/{total} files extracted...')

print('\nExtraction complete!')

Total files in zip: 465738
Progress: 0/465738 files extracted...
Progress: 5000/465738 files extracted...
Progress: 10000/465738 files extracted...
Progress: 15000/465738 files extracted...


In [ ]:
DATASET_ROOT = '/content/drive/MyDrive/A1/dataset'

# Paths from the project spec
CLASS_TRAIN = os.path.join(DATASET_ROOT, 'classification_data/train_data')
CLASS_VAL   = os.path.join(DATASET_ROOT, 'classification_data/val_data')
CLASS_TEST  = os.path.join(DATASET_ROOT, 'classification_data/test_data')
VERIF_DIR   = os.path.join(DATASET_ROOT, 'verification_data')
VERIF_PAIRS = os.path.join(DATASET_ROOT, 'verification_pairs_val.txt')

# Where processed (cropped + aligned) faces will be saved
PROCESSED_ROOT = '/content/drive/MyDrive/A1/processed/'
os.makedirs(PROCESSED_ROOT, exist_ok=True)

print('Paths configured successfully.')
print(f'  Dataset root   : {DATASET_ROOT}')
print(f'  Processed root : {PROCESSED_ROOT}')


#Load MTCNN Face Detector


In [ ]:
import torch
from facenet_pytorch import MTCNN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

mtcnn = MTCNN(
    image_size=112,   # all output faces → 112 x 112 px
    margin=20,        # padding added around the detected face box
    keep_all=False,   # keep only the largest / most confident face per image
    device=device
)

print('MTCNN face detector ready.')

#Detect, Crop & Save All Images


In [ ]:
from PIL import Image
from tqdm import tqdm
import numpy as np


def save_face_tensor(face_tensor, out_path):
    #Convert a (3, H, W) float tensor → PIL image → save to disk.
    face_np = face_tensor.permute(1, 2, 0).numpy()
    face_np = (
        (face_np - face_np.min()) /
        (face_np.max() - face_np.min()) * 255
    ).astype(np.uint8)
    Image.fromarray(face_np).save(out_path)


def process_classification_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    processed, skipped = 0, 0
    bad_paths = []

    identities = sorted(os.listdir(input_dir))
    for identity in tqdm(identities, desc=f'Processing {os.path.basename(input_dir)}'):
        in_id_dir  = os.path.join(input_dir, identity)
        out_id_dir = os.path.join(output_dir, identity)

        if not os.path.isdir(in_id_dir):
            continue
        os.makedirs(out_id_dir, exist_ok=True)

        for fname in os.listdir(in_id_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            img_path = os.path.join(in_id_dir, fname)
            out_path = os.path.join(out_id_dir, fname)

            try:
                img  = Image.open(img_path).convert('RGB')
                face = mtcnn(img)  # returns (3, 112, 112) tensor or None

                if face is not None:
                    save_face_tensor(face, out_path)
                    processed += 1
                else:
                    skipped += 1
                    bad_paths.append(img_path)
            except Exception as e:
                skipped += 1
                bad_paths.append(f'{img_path} — ERROR: {e}')

    return processed, skipped, bad_paths


#Run on all three classification splits
splits = [
    ('train_data', CLASS_TRAIN),
    ('val_data',   CLASS_VAL),
    ('test_data',  CLASS_TEST),
]

for split_name, in_dir in splits:
    out_dir = os.path.join(PROCESSED_ROOT, 'classification_data', split_name)
    p, s, bad = process_classification_folder(in_dir, out_dir)
    print(f'\n{split_name}: {p} processed, {s} skipped')
    if bad:
        print(f'  First skipped: {bad[:3]}', '...' if len(bad) > 3 else '')


#Run on verification images (flat folder, no identity subfolders)
VERIF_OUT = os.path.join(PROCESSED_ROOT, 'verification_data')
os.makedirs(VERIF_OUT, exist_ok=True)

verif_processed, verif_skipped = 0, 0
for fname in tqdm(os.listdir(VERIF_DIR), desc='Processing verification_data'):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    try:
        img  = Image.open(os.path.join(VERIF_DIR, fname)).convert('RGB')
        face = mtcnn(img)
        if face is not None:
            save_face_tensor(face, os.path.join(VERIF_OUT, fname))
            verif_processed += 1
        else:
            verif_skipped += 1
    except Exception as e:
        verif_skipped += 1
        print(f'  Error on {fname}: {e}')

print(f'\nverification_data: {verif_processed} processed, {verif_skipped} skipped')

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os


#Standard normalisation
TRANSFORM = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


class ClassificationDataset(Dataset):

    def __init__(self, root_dir, transform=TRANSFORM):
        self.transform = transform
        self.samples = []           # list of (image_path, label_int)
        self.identity_to_idx = {}   # maps folder name → integer label

        identities = sorted(os.listdir(root_dir))
        for idx, identity in enumerate(identities):
            id_dir = os.path.join(root_dir, identity)
            if not os.path.isdir(id_dir):
                continue
            self.identity_to_idx[identity] = idx
            for fname in os.listdir(id_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append(
                        (os.path.join(id_dir, fname), idx)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

    @property
    def num_classes(self):
        #Total number of unique identities needed for the softmax output head.
        return len(self.identity_to_idx)


class VerificationDataset(Dataset):

    #image pairs for face verification evaluation.
    def __init__(self, pairs_file, img_dir, transform=TRANSFORM):
        self.transform = transform
        self.img_dir = img_dir
        self.pairs = []

        with open(pairs_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 3:
                    img1, img2, label = parts
                    self.pairs.append((img1, img2, int(label)))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_name, img2_name, label = self.pairs[idx]
        img1 = Image.open(os.path.join(self.img_dir, img1_name)).convert('RGB')
        img2 = Image.open(os.path.join(self.img_dir, img2_name)).convert('RGB')
        return self.transform(img1), self.transform(img2), label

In [ ]:
PROC_TRAIN = os.path.join(PROCESSED_ROOT, 'classification_data/train_data')
PROC_VERIF = os.path.join(PROCESSED_ROOT, 'verification_data')

train_ds = ClassificationDataset(PROC_TRAIN)
verif_ds = VerificationDataset(VERIF_PAIRS, PROC_VERIF)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
verif_loader = DataLoader(verif_ds, batch_size=32, shuffle=False, num_workers=2)

print(f'Training samples     : {len(train_ds)}')
print(f'Number of identities : {train_ds.num_classes}')
print(f'Verification pairs   : {len(verif_ds)}')

# Peek at one batch
imgs, labels = next(iter(train_loader))
print(f'\nBatch shape : {imgs.shape}')      # should be [32, 3, 112, 112]
print(f'Label range : {labels.min().item()} – {labels.max().item()}')
print('\nAll checks passed. Phase 1 complete.')


#Visual Sanity Check


In [ ]:
import matplotlib.pyplot as plt


def denorm(tensor):
    #Reverse normalisation so images display with correct colours.
    return (tensor * 0.5 + 0.5).clamp(0, 1)


imgs, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(denorm(imgs[i]).permute(1, 2, 0))
    ax.set_title(f'ID: {labels[i].item()}', fontsize=8)
    ax.axis('off')

plt.suptitle('Sample training faces — after crop, align & normalise', fontsize=12)
plt.tight_layout()
plt.show()